# survey_processor — SURVEY-IQ → Acquire

Process raw **SURVEY-IQ** downhole geophysical exports into Acquire-ready format
and generate an inline gamma log figure.

**Run order:** `Kernel → Restart & Run All`

| Step | Cell | Output |
|------|------|--------|
| 1 | Configure paths | file-existence check |
| 2 | Helper functions | *(silent)* |
| 3 | Process CSV | DataFrame preview |
| 4 | Process LAS | change list |
| 5 | Gamma figure | inline plot + PNG |

**Outputs** (written to `processed/` next to the CSV):

| File | Description |
|------|-------------|
| `{HOLEID}_{DDMMYYYY}_REFLEX.csv` | Cleaned survey, CRLF line endings |
| `{HOLEID}_UP.las` | Cleaned LAS |
| `{HOLEID}_gamma.png` | Gamma log figure, 180 dpi |

In [ ]:
%matplotlib inline

import math
import re
import sys
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from IPython.display import display as _display

print("Dependencies loaded.")

## 1 — Configure paths

Edit the two variables below, then run this cell.

In [ ]:
# ── Set your file paths here ─────────────────────────────────────────────────
CSV_PATH = Path("BH001_15-03-2024_multishot.csv")   # SURVEY-IQ multishot CSV
LAS_PATH = Path("BH001_gamma.las")                   # SURVEY-IQ LAS gamma log
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = CSV_PATH.resolve().parent / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _check(p): return "✓" if p.is_file() else "✗  NOT FOUND"
print(f"CSV : {CSV_PATH}  {_check(CSV_PATH)}")
print(f"LAS : {LAS_PATH}  {_check(LAS_PATH)}")
print(f"Out : {OUTPUT_DIR}")

## 2 — Helper functions

Run the four cells below to define all processing functions.
No output is expected.

In [ ]:
# ── String / numeric utilities ────────────────────────────────────────────────

def strip_trailing_zeros(s):
    if "." in s:
        s = s.rstrip("0").rstrip(".")
    return s

_QC_EMPTY = frozenset({"", "0", "NA", "N/A", "nan", "NaN", "NONE", "None", "none"})

In [ ]:
# ── CSV helpers ───────────────────────────────────────────────────────────────

def fmt_earth_rate_delta(raw):
    # Round to 9 dp then strip trailing zeros.
    # SURVEY-IQ ERD values carry 16-18 dp of floating-point noise.
    s = str(raw).strip()
    if not s or s in ("nan", "NaN", "NA", "N/A"):
        return s
    try:
        r = Decimal(s).quantize(Decimal("0.000000001"), rounding=ROUND_HALF_UP)
        return strip_trailing_zeros(str(r))
    except InvalidOperation:
        return s


def fmt_strip_zeros_3dp(raw):
    # Round to 3 dp then strip trailing zeros.
    # Dip / Azimuth / Gravity TF / Vertical TF are zero-padded by SURVEY-IQ.
    s = str(raw).strip()
    if not s or s in ("nan", "NaN", "NA", "N/A"):
        return s
    try:
        r = Decimal(s).quantize(Decimal("0.001"), rounding=ROUND_HALF_UP)
        return strip_trailing_zeros(str(r))
    except InvalidOperation:
        return s


def count_qc_entries(value):
    # Count flag tokens in a QC cell (comma / pipe / semicolon separated).
    s = str(value).strip()
    if s in _QC_EMPTY:
        return 0
    return sum(1 for p in re.split(r"[,|;]", s)
               if p.strip() and p.strip() not in _QC_EMPTY)


def find_qc_column(df):
    # Return the QC column name, or None.
    exact = {"QC", "QC FLAGS", "QC FLAG", "QUALITY FLAGS", "QUALITY FLAG"}
    for col in df.columns:
        if col.strip().upper() in exact:
            return col
    for col in df.columns:
        if col.strip().upper().startswith("QC"):
            return col
    return None


def date_from_filename(name):
    # Extract DD-MM-YYYY from filename → return DDMMYYYY.
    m = re.search(r"(\d{2})-(\d{2})-(\d{4})", name)
    if not m:
        raise ValueError(
            f"No DD-MM-YYYY date found in {name!r}. "
            "Expected something like '15-03-2024'."
        )
    dd, mm, yyyy = m.groups()
    return f"{dd}{mm}{yyyy}"

In [ ]:
# ── LAS text helpers — constants + field parser ───────────────────────────────

_LOGU_BLANK_RE = re.compile(r"^(\s*LOGU\.\S*)(\s+)(:.*)")
_EXPORT_VER_RE = re.compile(r"^\s*EXPORTED\s+FROM\s+APP\s+VERSION\s*:", re.IGNORECASE)
_NOTES_RE      = re.compile(r"^\s*NOTES\s*:",                             re.IGNORECASE)


def parse_las_field(line):
    # Parse one LAS parameter/curve line → (mnem, unit, value, desc) or None.
    # LAS 2.0:  MNEM.UNIT  VALUE : DESCRIPTION
    # Unit is present only when a non-whitespace char follows the dot directly.
    stripped = line.strip()
    if not stripped or stripped.startswith("#"):
        return None
    colon = line.find(":")
    dot   = line.find(".")
    if colon == -1 or dot == -1 or dot > colon:
        return None
    mnem      = line[:dot].strip()
    after_dot = line[dot + 1 : colon]
    desc      = line[colon + 1 :].strip()
    if after_dot and after_dot[0] not in (" ", "\t"):
        parts = after_dot.split(None, 1)
        unit  = parts[0]
        value = parts[1].strip() if len(parts) > 1 else ""
    else:
        unit  = ""
        value = after_dot.strip()
    return mnem, unit, value, desc


def _patch_logu(line):
    # Insert WREGAM081 into a LOGU. line whose value field is blank.
    m = _LOGU_BLANK_RE.match(line)
    if m:
        return m.group(1) + "              WREGAM081 " + m.group(3)
    return line

In [ ]:
# ── LAS parser + gamma-column finder ─────────────────────────────────────────

def parse_las(text):
    # Parse a LAS 2.0 file → {metadata, curves, null, data}.
    lines    = text.splitlines()
    metadata = {}
    curves   = []
    data_lines = []
    in_meta = in_curve = in_data = False

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("~"):
            head = stripped.upper()
            in_meta  = head.startswith("~V") or head.startswith("~W") or head.startswith("~P")
            in_curve = head.startswith("~C")
            in_data  = head.startswith("~A")
            continue
        if stripped.startswith("#") or not stripped:
            continue
        if in_meta:
            p = parse_las_field(line)
            if p:
                metadata[p[0].strip().upper()] = p[2]
        elif in_curve:
            p = parse_las_field(line)
            if p:
                curves.append(p[0].strip().upper())
        elif in_data:
            data_lines.append(stripped)

    null_val = float(metadata.get("NULL", -999.25))
    arrays   = [[] for _ in range(len(curves))]
    for dl in data_lines:
        parts = dl.split()
        for i in range(min(len(curves), len(parts))):
            try:    arrays[i].append(float(parts[i]))
            except: arrays[i].append(float("nan"))

    data = {}
    for i, mnem in enumerate(curves):
        arr = np.array(arrays[i], dtype=float)
        tol = max(abs(null_val) * 1e-4, 0.01)
        arr[np.abs(arr - null_val) < tol] = np.nan
        data[mnem] = arr

    return {"metadata": metadata, "curves": curves, "null": null_val, "data": data}


def find_gamma_column(curves):
    # Return the gamma ray curve mnemonic.
    for cand in ("GR", "SGR", "CGR", "GAMMA", "NGAM", "GAPI", "GR_CORR", "GRC"):
        if cand in curves:
            return cand
    for c in curves:
        if "GR" in c or "GAMMA" in c:
            return c
    raise ValueError(f"No gamma ray curve found. Available: {curves}")

## 3 — Process CSV

In [ ]:
def process_csv(csv_path, output_dir):
    # Returns (output_path, hole_id, df_processed)
    date_str = date_from_filename(csv_path.name)

    # Read all columns as strings; keep literal 'NA' — do not coerce to NaN.
    df = pd.read_csv(csv_path, dtype=str, na_values=[], keep_default_na=False)

    # 1. Rename TN Azimuth → Azimuth
    if "TN Azimuth" in df.columns:
        df = df.rename(columns={"TN Azimuth": "Azimuth"})

    # 2. Strip leading RIG prefix only (RIG276→276; BDC276 unchanged)
    if "Rig" in df.columns:
        df["Rig"] = df["Rig"].str.replace(r"^RIG", "", regex=True)

    # 3. Remove duplicate Measured Depth rows
    depth_col, erd_col = "Measured Depth", "Earth Rate Delta"
    n_before = len(df)
    if depth_col in df.columns:
        qc_col = find_qc_column(df)
        df["__d"] = pd.to_numeric(df[depth_col], errors="coerce")
        df["__e"] = (pd.to_numeric(df[erd_col], errors="coerce").fillna(0.)
                     if erd_col in df.columns else pd.Series(0., index=df.index))
        df["__q"] = df[qc_col].apply(count_qc_entries) if qc_col else 0
        df = (df.sort_values(["__d","__q","__e"], ascending=True, kind="stable")
                .drop_duplicates(subset=[depth_col], keep="first"))
        df = (df.drop(columns=["__d","__e","__q"])
                .assign(__d=lambda d: pd.to_numeric(d[depth_col], errors="coerce"))
                .sort_values("__d", kind="stable")
                .drop(columns=["__d"])
                .reset_index(drop=True))

    # 4. Format Earth Rate Delta
    if erd_col in df.columns:
        df[erd_col] = df[erd_col].apply(fmt_earth_rate_delta)

    # 5. Strip trailing zeros from angular/toolface columns
    for col in ("Dip", "Azimuth", "Gravity TF", "Vertical TF"):
        if col in df.columns:
            df[col] = df[col].apply(fmt_strip_zeros_3dp)

    # 6. Extract hole ID
    hole_col = "Drillhole Name"
    if hole_col not in df.columns:
        raise ValueError(f"Column {hole_col!r} not found. Present: {list(df.columns)}")
    hole_id = df[hole_col].iloc[0].strip()

    # 7. Write with CRLF line endings (required by Acquire)
    out_path = output_dir / f"{hole_id}_{date_str}_REFLEX.csv"
    df.to_csv(out_path, index=False, lineterminator="\r\n")
    return out_path, hole_id, df, n_before

In [ ]:
csv_out, hole_id, df, n_before = process_csv(CSV_PATH, OUTPUT_DIR)
n_dropped = n_before - len(df)
print(f"Hole ID  : {hole_id!r}")
print(f"Rows in  : {n_before}  |  duplicate rows removed: {n_dropped}  |  rows out: {len(df)}")
print(f"Saved    : {csv_out}")
print()
df

## 4 — Process LAS

In [ ]:
def process_las(las_path, hole_id, output_dir):
    # Returns (output_path, [(action, text), ...])
    text  = las_path.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()

    in_param = in_other = False
    out_lines = []
    changes   = []

    for line in lines:
        head = line.strip().upper()
        if head.startswith("~"):
            in_param = head.startswith("~P")
            in_other = head.startswith("~O")
            out_lines.append(line)
            continue
        if in_param and re.match(r"\s*LOGU\.", line):
            patched = _patch_logu(line)
            if patched != line:
                changes.append(("LOGU populated", patched.strip()))
            line = patched
        if in_other:
            if _EXPORT_VER_RE.match(line):
                changes.append(("Removed", line.strip()))
                continue
            if _NOTES_RE.match(line):
                changes.append(("Removed", line.strip()))
                continue
        out_lines.append(line)

    out_text = "\n".join(out_lines)
    if text.endswith("\n"):
        out_text += "\n"

    out_path = output_dir / f"{hole_id}_UP.las"
    out_path.write_text(out_text, encoding="utf-8")
    return out_path, changes

In [ ]:
las_out, las_changes = process_las(LAS_PATH, hole_id, OUTPUT_DIR)
print(f"Saved: {las_out}")
print()
print("Changes applied:")
for action, text in las_changes:
    print(f"  [{action}]  {text}")

## 5 — Gamma log figure

X-axis (Gamma Ray, API) on **top**; depth increases **downward** from 0.
X-axis upper limit auto-scales to `ceil(max_GR / 10) × 10`.

In [ ]:
def generate_gamma_figure(las_path, hole_id, output_dir):
    # Returns (out_path, fig)
    text = las_path.read_text(encoding="utf-8", errors="replace")
    las  = parse_las(text)
    curves, data, metadata = las["curves"], las["data"], las["metadata"]

    if not curves:
        raise ValueError("No curve information found in LAS file.")

    depth_col = curves[0]                    # LAS convention: depth is always first
    gamma_col = find_gamma_column(curves)
    depth, gamma = data[depth_col], data[gamma_col]

    valid  = np.isfinite(depth) & np.isfinite(gamma)
    depth, gamma = depth[valid], gamma[valid]
    if depth.size == 0:
        raise ValueError("No valid depth/gamma data after null-value removal.")

    # X-axis: round max gamma up to nearest 10 — never clips high GR peaks
    x_max    = max(math.ceil(float(gamma.max()) / 10) * 10, 10)
    y_bottom = float(depth.max())
    y_top    = 0.0

    fig_h = max(6.0, min(24.0, (y_bottom - float(depth.min())) / 8.0))
    fig, ax = plt.subplots(figsize=(4.5, fig_h))

    # Curve: line + light fill
    ax.plot(gamma, depth, color="steelblue", linewidth=0.8)
    ax.fill_betweenx(depth, 0.0, gamma, alpha=0.12, color="steelblue")

    # X-axis on top
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set_xlabel("Gamma Ray (API)", labelpad=8)
    ax.set_xlim(0, x_max)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.grid(True, axis="x", linestyle="--", linewidth=0.5, color="gray", alpha=0.65)

    # Y-axis: 0 at top, depth increases downward — explicit ylim, not invert_yaxis()
    ax.set_ylim(y_bottom, y_top)
    ax.set_ylabel("Depth (m)")
    ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(5))
    ax.tick_params(axis="y", which="minor", length=3)

    # Title + subtitle from LAS metadata
    tool_name = (metadata.get("TOOL") or metadata.get("DEVI") or
                 metadata.get("GDEV") or metadata.get("BSEL") or
                 metadata.get("SRVC") or "")
    log_date  = (metadata.get("DATE") or metadata.get("LDAT") or
                 metadata.get("DDAT") or "")
    parts    = [p.strip() for p in (tool_name, log_date) if p.strip()]
    subtitle = "  |  ".join(parts)
    title    = f"{hole_id} — Gamma Ray"
    if subtitle:
        title = f"{title}\n{subtitle}"
    ax.set_title(title, fontsize=10, pad=50)

    out_path = output_dir / f"{hole_id}_gamma.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight", pad_inches=0.15)
    return out_path, fig

In [ ]:
png_out, fig = generate_gamma_figure(las_out, hole_id, OUTPUT_DIR)
plt.show()
plt.close(fig)
print(f"Saved: {png_out}")

## Summary

In [ ]:
print("Output files:")
for p in sorted(OUTPUT_DIR.glob(f"{hole_id}*")):
    kb = p.stat().st_size / 1024
    print(f"  {kb:6.1f} KB  {p.name}")